# Hybrid MPI-CUDA Distributed Stencil & Cross-Tier Cache-Line Coherency (Python)

### Cell 1 — Hardware & MPI environment setup

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!which mpicc > /dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null)
!pip install -q mpi4py numba numpy "numba-cuda[cu12]"

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-175f6f16-84a4-9268-9b73-ae90379d451f, 7.5, 15360 MiB
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuda-python 13.0.3 requires cuda-bindings~=13.0.3, but you have cuda-bindings 12.9.10 which is incompatible.
torch 2.14.1+cu130 requires cuda-bindings<14,>=13.0.3; platform_system == "Linux" and python_version < "3.15", but you have cuda-bindings 12.9.10 which is 

### Cell 2 — Solo student identity configuration

In [2]:
GITHUB_USERNAME = "mikazhan"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME}")

Verified solo student handle: mikazhan


### Cell 3 — Hybrid MPI-CUDA & shared-memory coherency engine

In [3]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import sys
import json
import math
import time
import hashlib
import platform
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, njit, float64, uint64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2
GLOBAL_NX = 512
GLOBAL_NY = 512
STENCIL_STEPS = 300
SHM_ITERS = 40_000_000
LAMBDA_DAMP = 0.15

PROC_DIMS = (2, 2)
REPORT_EVERY = 100
CFL_SAFETY = 0.8
KAPPA_MIN, KAPPA_MAX = 0.08, 0.24
N_PULSES = 3
PARITY_TOL = 1e-12
SHM_SLOTS = 64
SHM_ALIGN = 128
SHM_STRIDES = (1, 8, 16)
SHM_REPEATS = 3
MANIFEST_PATH = "hybrid_lab_manifest.json"
TAG_TO_UP, TAG_TO_DOWN, TAG_TO_LEFT, TAG_TO_RIGHT = 10, 11, 12, 13


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def get_gpu_name() -> str:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            encoding="utf-8",
        )
        return out.strip().splitlines()[0].strip()
    except Exception:
        return "unknown"


def detect_cpu_line_size() -> int:
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


def detect_cpu_topology():
    model, physical = platform.processor() or "unknown", None
    try:
        cores = set()
        phys = None
        with open("/proc/cpuinfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.lower().startswith("model name"):
                    model = line.split(":", 1)[1].strip()
                elif line.startswith("physical id"):
                    phys = line.split(":")[1].strip()
                elif line.startswith("core id"):
                    cores.add((phys, line.split(":")[1].strip()))
        physical = len(cores) or None
    except Exception:
        pass
    return model, physical


def log0(ctx, msg):
    if ctx["rank"] == 0:
        print(msg, flush=True)


# ==============================================================================
# TASK 1: Hybrid MPI Cartesian Grid + Node-Local MPI-3 Shared-Memory Window
# ==============================================================================
def init_hybrid_context(github_user: str) -> dict:
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")

    cuda.select_device(0)
    gpu_uuid = comm.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    gpu_name = comm.bcast(get_gpu_name() if rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}"
    line_bytes = detect_cpu_line_size()

    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}".encode("utf-8")).hexdigest()
    instance_seed = int(digest[:16], 16)

    cart_comm = comm.Create_cart(dims=list(PROC_DIMS), periods=[False, False], reorder=False)
    cart_rank = cart_comm.Get_rank()
    row_rank, col_rank = cart_comm.Get_coords(cart_rank)
    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    node_size = node_comm.Get_size()
    itemsize = np.dtype(np.uint64).itemsize
    nbytes = (SHM_SLOTS + SHM_ALIGN // itemsize) * itemsize if node_rank == 0 else 0
    win = MPI.Win.Allocate_shared(nbytes, itemsize, comm=node_comm)
    buf, _ = win.Shared_query(0)
    raw = np.ndarray(buffer=buf, dtype=np.uint64, shape=(SHM_SLOTS + SHM_ALIGN // itemsize,))
    offset = (-raw.ctypes.data % SHM_ALIGN) // itemsize
    shm_view = raw[offset:offset + SHM_SLOTS]
    assert shm_view.ctypes.data % 64 == 0

    cpu_model, physical_cores = detect_cpu_topology()
    ctx = dict(
        comm=comm, cart_comm=cart_comm, node_comm=node_comm, win=win, shm_view=shm_view,
        rank=cart_rank, size=size, node_rank=node_rank, node_size=node_size,
        coords=(row_rank, col_rank),
        nbr_up=nbr_up, nbr_down=nbr_down, nbr_left=nbr_left, nbr_right=nbr_right,
        gpu_uuid=gpu_uuid, gpu_name=gpu_name, cpu_sig=cpu_sig, cpu_model=cpu_model,
        logical_cores=os.cpu_count(), physical_cores=physical_cores,
        cpu_line_bytes=line_bytes, instance_seed=instance_seed, github_user=github_user.strip(),
        ny=GLOBAL_NY // PROC_DIMS[0], nx=GLOBAL_NX // PROC_DIMS[1],
        shm_base_addr=int(shm_view.ctypes.data),
    )

    def fmt(n):
        return "PROC_NULL" if n == MPI.PROC_NULL else str(n)

    for r in range(size):
        cart_comm.Barrier()
        if r == cart_rank:
            print(f"[rank {cart_rank}] coords=({row_rank},{col_rank}) node_rank={node_rank}/{node_size} "
                  f"up={fmt(nbr_up)} down={fmt(nbr_down)} left={fmt(nbr_left)} right={fmt(nbr_right)}",
                  flush=True)
    cart_comm.Barrier()
    log0(ctx, f"[init] MPI ranks={size}, Cartesian dims={list(PROC_DIMS)}, periods=[False, False], node_comm size={node_size}")
    log0(ctx, f"[init] GPU={gpu_name} UUID={gpu_uuid}")
    log0(ctx, f"[init] CPU={cpu_model} logical={os.cpu_count()} physical={physical_cores} line={line_bytes} B sig={cpu_sig}")
    log0(ctx, f"[init] GITHUB_USERNAME={ctx['github_user']} INSTANCE_SEED={instance_seed} (0x{instance_seed:016x})")
    log0(ctx, f"[init] shm window: {SHM_SLOTS} uint64 slots, base=0x{ctx['shm_base_addr']:x} "
              f"(mod 64 = {ctx['shm_base_addr'] % 64}, mod 128 = {ctx['shm_base_addr'] % 128})")
    return ctx


# ==============================================================================
# TASK 2: Rank-Local GPU Subdomain & Halo Pack/Unpack Kernels
# ==============================================================================
def seeded_parameters(instance_seed: int):
    rng = np.random.default_rng(instance_seed)
    return dict(
        kappa_phase=rng.uniform(0.0, 1.0, size=2),
        kappa_freq=rng.integers(1, 4, size=2),
        pulse_cx=rng.uniform(0.2, 0.8, size=N_PULSES),
        pulse_cy=rng.uniform(0.2, 0.8, size=N_PULSES),
        pulse_amp=rng.uniform(0.8, 1.5, size=N_PULSES),
        pulse_sigma=rng.uniform(0.05, 0.10, size=N_PULSES),
        src_cx=rng.uniform(0.3, 0.7),
        src_cy=rng.uniform(0.3, 0.7),
        src_amp=rng.uniform(0.01, 0.03),
        src_sigma=0.03,
    )


def physical_fields(X, Y, p):
    mid = 0.5 * (KAPPA_MIN + KAPPA_MAX)
    amp = 0.5 * (KAPPA_MAX - KAPPA_MIN)
    kappa = mid + amp * (np.sin(2.0 * np.pi * (p["kappa_freq"][0] * X + p["kappa_phase"][0]))
                         * np.cos(2.0 * np.pi * (p["kappa_freq"][1] * Y + p["kappa_phase"][1])))
    kappa = np.clip(kappa, KAPPA_MIN, KAPPA_MAX)
    u0 = np.zeros_like(X)
    for k in range(N_PULSES):
        r2 = (X - p["pulse_cx"][k]) ** 2 + (Y - p["pulse_cy"][k]) ** 2
        u0 += p["pulse_amp"][k] * np.exp(-r2 / (2.0 * p["pulse_sigma"][k] ** 2))
    r2s = (X - p["src_cx"]) ** 2 + (Y - p["src_cy"]) ** 2
    S = p["src_amp"] * np.exp(-r2s / (2.0 * p["src_sigma"] ** 2))
    return kappa, u0, S


def initialize_fields(ctx):
    ny, nx = ctx["ny"], ctx["nx"]
    row_rank, col_rank = ctx["coords"]
    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)
    gi = row_rank * ny + np.arange(ny + 2, dtype=np.float64)
    gj = col_rank * nx + np.arange(nx + 2, dtype=np.float64)
    Y, X = np.meshgrid(gi * dy, gj * dx, indexing="ij")

    p = seeded_parameters(ctx["instance_seed"])
    kappa, u0, S = physical_fields(X, Y, p)
    u0[0, :] = u0[ny + 1, :] = 0.0
    u0[:, 0] = u0[:, nx + 1] = 0.0
    S[0, :] = S[ny + 1, :] = 0.0
    S[:, 0] = S[:, nx + 1] = 0.0
    u0, kappa, S = (np.ascontiguousarray(a) for a in (u0, kappa, S))

    kappa_max = ctx["cart_comm"].allreduce(float(kappa[1:ny + 1, 1:nx + 1].max()), op=MPI.MAX)
    kappa_min = ctx["cart_comm"].allreduce(float(kappa[1:ny + 1, 1:nx + 1].min()), op=MPI.MIN)
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max
    phys = dict(dx=dx, dy=dy, dt=dt, dt_max=dt_max, kappa_max=kappa_max, kappa_min=kappa_min,
                params={k: (v.tolist() if isinstance(v, np.ndarray) else float(v)) for k, v in p.items()})
    log0(ctx, f"[fields] local padded shape={u0.shape}, dx=dy={dx:.6e}, kappa in [{kappa_min:.5f}, {kappa_max:.5f}]")
    log0(ctx, f"[fields] dt_max(CFL)={dt_max:.6e}, dt={dt:.6e} (safety {CFL_SAFETY})")
    return u0, kappa, S, phys


@cuda.jit
def pack_halos_kernel(u, send_buf, ny, nx):
    k = cuda.grid(1)
    if k < nx:
        send_buf[k] = u[1, k + 1]
        send_buf[nx + k] = u[ny, k + 1]
    if k < ny:
        send_buf[2 * nx + k] = u[k + 1, 1]
        send_buf[2 * nx + ny + k] = u[k + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, recv_buf, ny, nx):
    k = cuda.grid(1)
    if k < nx:
        u[0, k + 1] = recv_buf[k]
        u[ny + 1, k + 1] = recv_buf[nx + k]
    if k < ny:
        u[k + 1, 0] = recv_buf[2 * nx + k]
        u[k + 1, nx + 1] = recv_buf[2 * nx + ny + k]


class HaloBuffers:
    def __init__(self, ny, nx):
        self.ny, self.nx = ny, nx
        n = 2 * nx + 2 * ny
        self.d_send = cuda.device_array(n, dtype=np.float64)
        self.d_recv = cuda.to_device(np.zeros(n, dtype=np.float64))
        self.h_send = cuda.pinned_array(n, dtype=np.float64)
        self.h_recv = cuda.pinned_array(n, dtype=np.float64)
        self.h_send[:] = 0.0
        self.h_recv[:] = 0.0
        self.threads = 256
        self.blocks = (max(nx, ny) + self.threads - 1) // self.threads

    def segs(self, buf):
        nx, ny = self.nx, self.ny
        return dict(top=buf[0:nx], bottom=buf[nx:2 * nx],
                    left=buf[2 * nx:2 * nx + ny], right=buf[2 * nx + ny:2 * nx + 2 * ny])


def post_halo_exchange(ctx, bufs, d_u, stream_boundary):
    pack_halos_kernel[bufs.blocks, bufs.threads, stream_boundary](d_u, bufs.d_send, bufs.ny, bufs.nx)
    bufs.d_send.copy_to_host(bufs.h_send, stream=stream_boundary)
    stream_boundary.synchronize()
    cart = ctx["cart_comm"]
    s, r = bufs.segs(bufs.h_send), bufs.segs(bufs.h_recv)
    return [
        cart.Irecv([r["top"], MPI.DOUBLE], source=ctx["nbr_up"], tag=TAG_TO_DOWN),
        cart.Irecv([r["bottom"], MPI.DOUBLE], source=ctx["nbr_down"], tag=TAG_TO_UP),
        cart.Irecv([r["left"], MPI.DOUBLE], source=ctx["nbr_left"], tag=TAG_TO_RIGHT),
        cart.Irecv([r["right"], MPI.DOUBLE], source=ctx["nbr_right"], tag=TAG_TO_LEFT),
        cart.Isend([s["top"], MPI.DOUBLE], dest=ctx["nbr_up"], tag=TAG_TO_UP),
        cart.Isend([s["bottom"], MPI.DOUBLE], dest=ctx["nbr_down"], tag=TAG_TO_DOWN),
        cart.Isend([s["left"], MPI.DOUBLE], dest=ctx["nbr_left"], tag=TAG_TO_LEFT),
        cart.Isend([s["right"], MPI.DOUBLE], dest=ctx["nbr_right"], tag=TAG_TO_RIGHT),
    ]


def complete_halo_exchange(bufs, reqs, d_u, stream_boundary):
    MPI.Request.Waitall(reqs)
    bufs.d_recv.copy_to_device(bufs.h_recv, stream=stream_boundary)
    unpack_halos_kernel[bufs.blocks, bufs.threads, stream_boundary](d_u, bufs.d_recv, bufs.ny, bufs.nx)


# ==============================================================================
# TASK 3: 2D Shared-Memory Tiled Stencil & Dual-Stream Pipeline
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kappa, src, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    smem = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    sy = ty + 1
    sx = tx + 1
    lo_r = r_start - 1
    hi_r = r_end + 1
    lo_c = c_start - 1
    hi_c = c_end + 1

    if i <= hi_r and j <= hi_c:
        smem[sy, sx] = u[i, j]
    if ty == 0 and i - 1 >= lo_r and j <= hi_c:
        smem[0, sx] = u[i - 1, j]
    if ty == TILE_DIM - 1 and i + 1 <= hi_r and j <= hi_c:
        smem[SHARED_DIM - 1, sx] = u[i + 1, j]
    if tx == 0 and j - 1 >= lo_c and i <= hi_r:
        smem[sy, 0] = u[i, j - 1]
    if tx == TILE_DIM - 1 and j + 1 <= hi_c and i <= hi_r:
        smem[sy, SHARED_DIM - 1] = u[i, j + 1]

    cuda.syncthreads()

    if i <= r_end and j <= c_end:
        uc = smem[sy, sx]
        u_im = smem[sy - 1, sx]
        u_ip = smem[sy + 1, sx]
        u_jm = smem[sy, sx - 1]
        u_jp = smem[sy, sx + 1]
        kc = kappa[i, j]
        k_ip = 0.5 * (kappa[i + 1, j] + kc)
        k_im = 0.5 * (kappa[i - 1, j] + kc)
        k_jp = 0.5 * (kappa[i, j + 1] + kc)
        k_jm = 0.5 * (kappa[i, j - 1] + kc)
        div_i = (k_ip * (u_ip - uc) - k_im * (uc - u_im)) * inv_dx2
        div_j = (k_jp * (u_jp - uc) - k_jm * (uc - u_jm)) * inv_dy2
        u_new[i, j] = uc + dt * (div_i + div_j - lam * uc * uc * uc + src[i, j])


def launch_stencil(st, box, stream, phys):
    r0, r1, c0, c1 = box
    grid = ((c1 - c0 + TILE_DIM) // TILE_DIM, (r1 - r0 + TILE_DIM) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        st["u"], st["un"], st["k"], st["s"], r0, r1, c0, c1,
        phys["dt"], 1.0 / (phys["dx"] ** 2), 1.0 / (phys["dy"] ** 2), LAMBDA_DAMP)


def make_state(u0, kappa, S, ny, nx):
    return dict(
        u=cuda.to_device(u0), un=cuda.to_device(u0), k=cuda.to_device(kappa), s=cuda.to_device(S),
        stream_boundary=cuda.stream(), stream_interior=cuda.stream(),
        full=(1, ny, 1, nx), interior=(2, ny - 1, 2, nx - 1),
        ring=[(1, 1, 1, nx), (ny, ny, 1, nx), (2, ny - 1, 1, 1), (2, ny - 1, nx, nx)],
    )


def step_serialized(ctx, st, bufs, phys):
    sb = st["stream_boundary"]
    reqs = post_halo_exchange(ctx, bufs, st["u"], sb)
    complete_halo_exchange(bufs, reqs, st["u"], sb)
    launch_stencil(st, st["full"], sb, phys)
    sb.synchronize()
    st["u"], st["un"] = st["un"], st["u"]


def step_overlapped(ctx, st, bufs, phys):
    sb, si = st["stream_boundary"], st["stream_interior"]
    reqs = post_halo_exchange(ctx, bufs, st["u"], sb)
    launch_stencil(st, st["interior"], si, phys)
    complete_halo_exchange(bufs, reqs, st["u"], sb)
    for box in st["ring"]:
        launch_stencil(st, box, sb, phys)
    sb.synchronize()
    si.synchronize()
    st["u"], st["un"] = st["un"], st["u"]


def global_energy(ctx, st, phys):
    ny, nx = ctx["ny"], ctx["nx"]
    interior = st["u"].copy_to_host()[1:ny + 1, 1:nx + 1]
    cart = ctx["cart_comm"]
    energy = cart.allreduce(float(interior.sum()) * phys["dx"] * phys["dy"], op=MPI.SUM)
    l2 = math.sqrt(cart.allreduce(float((interior ** 2).sum()), op=MPI.SUM) * phys["dx"] * phys["dy"])
    umax = cart.allreduce(float(np.abs(interior).max()), op=MPI.MAX)
    return energy, l2, umax


def run_stencil_mode(ctx, mode, u0, kappa, S, phys, bufs):
    st = make_state(u0, kappa, S, ctx["ny"], ctx["nx"])
    step_fn = step_serialized if mode == "serialized" else step_overlapped
    trace = []
    e, l2, m = global_energy(ctx, st, phys)
    trace.append(dict(step=0, energy=e, l2_norm=l2, max_abs_u=m))
    log0(ctx, f"[{mode:>10}] step {0:4d}  E={e:.12e}  ||u||2={l2:.6e}  max|u|={m:.6f}")
    wall = 0.0
    ev_ms = 0.0
    for start in range(0, STENCIL_STEPS, REPORT_EVERY):
        chunk = min(REPORT_EVERY, STENCIL_STEPS - start)
        cuda.synchronize()
        ctx["cart_comm"].Barrier()
        ev0, ev1 = cuda.event(), cuda.event()
        t0 = MPI.Wtime()
        ev0.record()
        for _ in range(chunk):
            step_fn(ctx, st, bufs, phys)
        ev1.record()
        ev1.synchronize()
        wall += MPI.Wtime() - t0
        try:
            ev_ms += cuda.event_elapsed_time(ev0, ev1)
        except Exception:
            pass
        e, l2, m = global_energy(ctx, st, phys)
        trace.append(dict(step=start + chunk, energy=e, l2_norm=l2, max_abs_u=m))
        log0(ctx, f"[{mode:>10}] step {start + chunk:4d}  E={e:.12e}  ||u||2={l2:.6e}  max|u|={m:.6f}")
    wall_ms = ctx["cart_comm"].allreduce(wall * 1e3, op=MPI.MAX)
    ev_ms = ctx["cart_comm"].allreduce(ev_ms, op=MPI.MAX)
    return dict(trace=trace, wall_ms=wall_ms, event_ms=ev_ms, final=st["u"].copy_to_host())


def ptx_audit():
    import re
    try:
        asm = tiled_stencil_kernel.inspect_asm()
    except Exception as exc:
        return dict(available=False, error=repr(exc), st_shared_f64=0, ld_shared_f64=0, bar_sync=0, passed=False)
    ptx = "\n".join(asm.values()) if isinstance(asm, dict) else str(asm)
    st_n = len(re.findall(r"\bst\.shared\.f64\b", ptx))
    ld_n = len(re.findall(r"\bld\.shared\.f64\b", ptx))
    st_any = len(re.findall(r"\bst\.shared\.", ptx))
    ld_any = len(re.findall(r"\bld\.shared\.", ptx))
    bar = len(re.findall(r"\bbar\.sync\b", ptx)) + len(re.findall(r"\bbarrier\.sync\b", ptx))
    return dict(available=True, st_shared_f64=st_n, ld_shared_f64=ld_n, st_shared_any=st_any,
                ld_shared_any=ld_any, bar_sync=bar, passed=bool(st_any > 0 and ld_any > 0 and bar > 0))


# ==============================================================================
# TASK 4: Multi-Rank Host Cache-Line False Sharing Sweep
# ==============================================================================
@njit(nogil=True, cache=False)
def host_shm_accumulate(shm_view, slot, iters, seed):
    s = uint64(seed)
    j = slot
    for i in range(iters):
        v = shm_view[j]
        nv = v + ((s ^ uint64(i)) & uint64(7)) + uint64(1)
        shm_view[j] = nv
        j = slot + np.int64(nv >> uint64(60))
    return j


def expected_counter(seed, iters):
    total = np.uint64(0)
    chunk = 10_000_000
    for a in range(0, iters, chunk):
        i = np.arange(a, min(a + chunk, iters), dtype=np.uint64)
        total += (((np.uint64(seed) ^ i) & np.uint64(7)) + np.uint64(1)).sum(dtype=np.uint64)
    return int(total)


def benchmark_shm_strides(ctx, rank_seed):
    node_comm, win, view = ctx["node_comm"], ctx["win"], ctx["shm_view"]
    rank = ctx["node_rank"]
    line = ctx["cpu_line_bytes"]
    base = ctx["shm_base_addr"]
    expected = expected_counter(rank_seed, SHM_ITERS)

    host_shm_accumulate(np.zeros(8, dtype=np.uint64), 0, 1000, 1)
    node_comm.Barrier()

    results = {}
    parity = {}
    configs = [("solo_stride1", 1, True)] + [(f"stride_{s}", s, False) for s in SHM_STRIDES]
    for name, stride, solo in configs:
        slot = rank * stride
        reps = []
        for _ in range(SHM_REPEATS):
            win.Fence()
            if rank == 0:
                view[:] = 0
            win.Fence()
            node_comm.Barrier()
            t0 = time.perf_counter_ns()
            if (not solo) or rank == 0:
                host_shm_accumulate(view, slot, SHM_ITERS, rank_seed)
            t1 = time.perf_counter_ns()
            node_comm.Barrier()
            t2 = time.perf_counter_ns()
            win.Fence()
            own = t1 - t0
            reps.append(dict(rank_kernel_ns_max=node_comm.allreduce(own, op=MPI.MAX),
                             barrier_to_barrier_ns=node_comm.allreduce(t2 - t0, op=MPI.MAX)))
            ok = (int(view[slot]) == expected) if ((not solo) or rank == 0) else True
            parity_ok = node_comm.allreduce(int(ok), op=MPI.MIN) == 1
        med = sorted(r["rank_kernel_ns_max"] for r in reps)[len(reps) // 2]
        slots = [r * stride for r in range(ctx["node_size"])] if not solo else [0]
        lines64 = sorted({(base + 8 * s) // 64 - base // 64 for s in slots})
        lines_native = sorted({(base + 8 * s) // line - base // line for s in slots})
        results[name] = dict(stride=stride, byte_spacing=8 * stride, solo=solo, slots=slots,
                             cpu_lines_64B=lines64, cpu_lines_native=lines_native,
                             latency_ns_median=med, ns_per_update=med / SHM_ITERS, repeats=reps)
        parity[name] = parity_ok
        log0(ctx, f"[shm {name:<13}] {med:>14,d} ns  {med/SHM_ITERS:7.3f} ns/update  "
                  f"slots={slots} lines(64B)={lines64}  parity={parity_ok}")
    s1 = results["stride_1"]["latency_ns_median"]
    s8 = results["stride_8"]["latency_ns_median"]
    s16 = results["stride_16"]["latency_ns_median"]
    ratios = dict(false_sharing_slowdown_stride1_vs_stride8=s1 / s8,
                  stride1_vs_stride16=s1 / s16, stride8_vs_stride16=s8 / s16)
    log0(ctx, f"[shm ratios] stride1/stride8={ratios['false_sharing_slowdown_stride1_vs_stride8']:.3f}x  "
              f"stride1/stride16={ratios['stride1_vs_stride16']:.3f}x  stride8/stride16={ratios['stride8_vs_stride16']:.3f}x")
    return dict(results=results, ratios=ratios, parity=parity, parity_passed=all(parity.values()),
                expected_counter_rank0=str(node_comm.bcast(expected, root=0)))


def llvm_audit():
    import re
    llvm = host_shm_accumulate.inspect_llvm()
    ir = "\n".join(llvm.values()) if isinstance(llvm, dict) else str(llvm)
    blocks, cur = [], None
    for line in ir.splitlines():
        if line.startswith("define "):
            cur = [line]
        elif cur is not None:
            cur.append(line)
            if line.startswith("}"):
                blocks.append("\n".join(cur))
                cur = None
    fn = ""
    for b in blocks:
        head = b.splitlines()[0]
        if "host_shm_accumulate" in head and "cpython" not in head and "cfunc" not in head:
            fn = b
            break
    loads = len(re.findall(r"=\s*load\s+i64", fn))
    stores = len(re.findall(r"\bstore\s+i64\b", fn))
    py_calls = sorted(set(re.findall(r"@(Py\w+|_Py\w+|NRT_\w+)", fn)))
    nogil = bool(host_shm_accumulate.targetoptions.get("nogil", False))
    return dict(function_found=bool(fn), load_i64=loads, store_i64=stores, python_object_calls=py_calls,
                nogil=nogil, store_i64_lines=[l.strip() for l in fn.splitlines() if re.search(r"\bstore\s+i64\b", l)][:6],
                passed=bool(fn and loads > 0 and stores > 0 and not py_calls and nogil))


# ==============================================================================
# TASK 5: Cross-Tier Verification & Manifest Export
# ==============================================================================
def main(github_user: str):
    ctx = init_hybrid_context(github_user)
    u0, kappa, S, phys = initialize_fields(ctx)
    bufs = HaloBuffers(ctx["ny"], ctx["nx"])

    warm = make_state(u0, kappa, S, ctx["ny"], ctx["nx"])
    step_overlapped(ctx, warm, bufs, phys)
    step_serialized(ctx, warm, bufs, phys)
    cuda.synchronize()
    ctx["cart_comm"].Barrier()
    log0(ctx, f"[gpu] kernels compiled; running {STENCIL_STEPS} steps per mode")

    ser = run_stencil_mode(ctx, "serialized", u0, kappa, S, phys, bufs)
    ovl = run_stencil_mode(ctx, "overlapped", u0, kappa, S, phys, bufs)
    parity_max_diff = ctx["cart_comm"].allreduce(float(np.abs(ser["final"] - ovl["final"]).max()), op=MPI.MAX)
    gpu_parity_ok = parity_max_diff < PARITY_TOL
    energies = [t["energy"] for t in ovl["trace"]]
    energy_monotonic = all(energies[k + 1] <= energies[k] for k in range(len(energies) - 1))
    traces_match = all(abs(a["energy"] - b["energy"]) < 1e-14 for a, b in zip(ser["trace"], ovl["trace"]))
    ptx = ctx["cart_comm"].bcast(ptx_audit() if ctx["rank"] == 0 else None, root=0)

    rank_seed = int(np.random.default_rng(ctx["instance_seed"]).integers(1, 2**31 - 1, size=4)[ctx["node_rank"]])
    shm = benchmark_shm_strides(ctx, rank_seed)
    llvm = ctx["cart_comm"].bcast(llvm_audit() if ctx["rank"] == 0 else None, root=0)

    topo = np.zeros((ctx["size"], 6), dtype=np.int64)
    topo[ctx["rank"]] = [ctx["coords"][0], ctx["coords"][1], ctx["nbr_up"], ctx["nbr_down"],
                         ctx["nbr_left"], ctx["nbr_right"]]
    topo_all = np.zeros_like(topo)
    ctx["cart_comm"].Allreduce(topo, topo_all, op=MPI.SUM)

    verification_passed = bool(gpu_parity_ok and ptx["passed"] and llvm["passed"]
                               and shm["parity_passed"] and energy_monotonic)

    if ctx["rank"] == 0:
        print("=" * 78)
        print(f"[gpu bench] gpu_serialized_ms = {ser['wall_ms']:.3f}  (CUDA events {ser['event_ms']:.3f})")
        print(f"[gpu bench] gpu_overlapped_ms = {ovl['wall_ms']:.3f}  (CUDA events {ovl['event_ms']:.3f})")
        print(f"[gpu bench] overlap speedup = {ser['wall_ms'] / ovl['wall_ms']:.4f}x")
        print(f"[gpu parity] max |serialized - overlapped| = {parity_max_diff:.3e} -> {'PASS' if gpu_parity_ok else 'FAIL'}")
        print(f"[energy] monotonic={energy_monotonic}  serialized/overlapped traces identical={traces_match}")
        print(f"[ptx] st.shared.f64={ptx['st_shared_f64']} ld.shared.f64={ptx['ld_shared_f64']} "
              f"bar.sync={ptx['bar_sync']} -> {'PASS' if ptx['passed'] else 'FAIL'}")
        print(f"[llvm] load i64={llvm['load_i64']} store i64={llvm['store_i64']} "
              f"python_object_calls={llvm['python_object_calls']} nogil={llvm['nogil']} -> {'PASS' if llvm['passed'] else 'FAIL'}")
        for l in llvm["store_i64_lines"][:4]:
            print(f"[llvm]   {l}")
        print(f"[shm parity] {shm['parity']}")
        print(f"[verification_passed] {verification_passed}")
        print("=" * 78)

        try:
            with open(os.path.abspath(__file__), "rb") as fh:
                src_hash = hashlib.sha256(fh.read()).hexdigest()
        except Exception:
            src_hash = None
        import numba
        manifest = dict(
            github_username=ctx["github_user"],
            gpu_uuid=ctx["gpu_uuid"], gpu_name=ctx["gpu_name"],
            cpu_sig=ctx["cpu_sig"], cpu_model=ctx["cpu_model"],
            logical_cores=ctx["logical_cores"], physical_cores=ctx["physical_cores"],
            cpu_line_bytes=ctx["cpu_line_bytes"],
            instance_seed=str(ctx["instance_seed"]), instance_seed_hex=f"0x{ctx['instance_seed']:016x}",
            mpi_ranks=ctx["size"], cart_dims=list(PROC_DIMS), cart_periods=[False, False],
            node_comm_size=ctx["node_size"], mpi_proc_null=int(MPI.PROC_NULL),
            topology=[dict(rank=r, coords=[int(topo_all[r, 0]), int(topo_all[r, 1])],
                           up=int(topo_all[r, 2]), down=int(topo_all[r, 3]),
                           left=int(topo_all[r, 4]), right=int(topo_all[r, 5])) for r in range(ctx["size"])],
            global_grid=[GLOBAL_NY, GLOBAL_NX], local_padded_shape=[ctx["ny"] + 2, ctx["nx"] + 2],
            tile_dim=TILE_DIM, shared_dim=SHARED_DIM, stencil_steps=STENCIL_STEPS, lambda_damp=LAMBDA_DAMP,
            dx=phys["dx"], dy=phys["dy"], dt=phys["dt"], dt_max_cfl=phys["dt_max"],
            kappa_max=phys["kappa_max"], kappa_min=phys["kappa_min"], seeded_params=phys["params"],
            energy_trace=ovl["trace"], energy_trace_serialized=ser["trace"],
            energy_monotonic=energy_monotonic, energy_traces_identical=traces_match,
            gpu_serialized_ms=ser["wall_ms"], gpu_overlapped_ms=ovl["wall_ms"],
            gpu_serialized_event_ms=ser["event_ms"], gpu_overlapped_event_ms=ovl["event_ms"],
            gpu_overlap_speedup=ser["wall_ms"] / ovl["wall_ms"],
            parity_max_diff=parity_max_diff, gpu_parity_passed=gpu_parity_ok,
            ptx_audit=ptx, ptx_audit_passed=ptx["passed"],
            shm_window=dict(slots=SHM_SLOTS, alignment_bytes=SHM_ALIGN, base_mod_64=ctx["shm_base_addr"] % 64,
                            base_mod_128=ctx["shm_base_addr"] % 128, iters_per_rank=SHM_ITERS,
                            repeats=SHM_REPEATS),
            shm_results=shm["results"], shm_ratios=shm["ratios"],
            shm_parity=shm["parity"], shm_parity_passed=shm["parity_passed"],
            shm_expected_counter_rank0=shm["expected_counter_rank0"],
            llvm_audit=llvm, llvm_audit_passed=llvm["passed"],
            verification_passed=verification_passed,
            tasks_completed=[1, 2, 3, 4, 5],
            versions=dict(python=platform.python_version(), numpy=np.__version__, numba=numba.__version__,
                          mpi=MPI.Get_library_version().splitlines()[0]),
            source_sha256=src_hash,
            timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        )
        with open(MANIFEST_PATH, "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, indent=2)
        print(f"[manifest] wrote {MANIFEST_PATH}")

    ctx["cart_comm"].Barrier()
    ctx["win"].Free()


if __name__ == "__main__":
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        raise SystemExit("usage: mpirun -np 4 python3 hybrid_mpi_cuda_cache.py <github_username>")
    main(sys.argv[1].strip())

Writing hybrid_mpi_cuda_cache.py


### Cell 4 — Multi-rank hybrid execution

In [4]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "hybrid_mpi_cuda_cache.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")

[rank 0] coords=(0,0) node_rank=0/4 up=PROC_NULL down=2 left=PROC_NULL right=1
[rank 1] coords=(0,1) node_rank=1/4 up=PROC_NULL down=3 left=0 right=PROC_NULL
[rank 2] coords=(1,0) node_rank=2/4 up=0 down=PROC_NULL left=PROC_NULL right=3
[rank 3] coords=(1,1) node_rank=3/4 up=1 down=PROC_NULL left=2 right=PROC_NULL
[init] MPI ranks=4, Cartesian dims=[2, 2], periods=[False, False], node_comm size=4
[init] GPU=Tesla T4, 7.5 UUID=GPU-175f6f16-84a4-9268-9b73-ae90379d451f
[init] CPU=Intel(R) Xeon(R) CPU @ 2.00GHz logical=2 physical=1 line=64 B sig=x86_64::2
[init] GITHUB_USERNAME=mikazhan INSTANCE_SEED=17370736032954436743 (0xf11140634b53cc87)
[init] shm window: 64 uint64 slots, base=0x7a24a7003180 (mod 64 = 0, mod 128 = 0)
[fields] local padded shape=(258, 258), dx=dy=1.949318e-03, kappa in [0.08000, 0.24000]
[fields] dt_max(CFL)=3.958167e-06, dt=3.166533e-06 (safety 0.8)
[gpu] kernels compiled; running 300 steps per mode
[serialized] step    0  E=1.063782502578e-01  ||u||2=2.580389e-01  ma

### Telemetry summary

In [5]:
import json
m = json.load(open("hybrid_lab_manifest.json"))
print(f"user={m['github_username']} seed={m['instance_seed']} ({m['instance_seed_hex']})")
print(f"GPU={m['gpu_name']} {m['gpu_uuid']}")
print(f"CPU={m['cpu_model']} logical={m['logical_cores']} physical={m['physical_cores']} line={m['cpu_line_bytes']} B")
print(f"PTX: st.shared.f64={m['ptx_audit']['st_shared_f64']} ld.shared.f64={m['ptx_audit']['ld_shared_f64']} bar.sync={m['ptx_audit']['bar_sync']}")
print(f"gpu_serialized_ms={m['gpu_serialized_ms']:.3f} gpu_overlapped_ms={m['gpu_overlapped_ms']:.3f} speedup={m['gpu_overlap_speedup']:.4f}x parity={m['parity_max_diff']:.3e}")
for k, r in m['shm_results'].items():
    print(f"shm {k:<13} {r['latency_ns_median']:>14,d} ns  {r['ns_per_update']:.3f} ns/update  lines(64B)={r['cpu_lines_64B']}")
print("shm ratios:", {k: round(v, 4) for k, v in m['shm_ratios'].items()})
a = m['llvm_audit']
print(f"LLVM: load i64={a['load_i64']} store i64={a['store_i64']} py_calls={a['python_object_calls']} nogil={a['nogil']}")
print(f"kappa_max={m['kappa_max']:.6f} dt_max={m['dt_max_cfl']:.6e} dt={m['dt']:.6e}")
for t in m['energy_trace']:
    print(f"  step {t['step']:4d}: E={t['energy']:.12e}  max|u|={t['max_abs_u']:.6f}")
print(f"verification_passed={m['verification_passed']}")

user=mikazhan seed=17370736032954436743 (0xf11140634b53cc87)
GPU=Tesla T4, 7.5 GPU-175f6f16-84a4-9268-9b73-ae90379d451f
CPU=Intel(R) Xeon(R) CPU @ 2.00GHz logical=2 physical=1 line=64 B
PTX: st.shared.f64=5 ld.shared.f64=5 bar.sync=1
gpu_serialized_ms=733.965 gpu_overlapped_ms=1515.857 speedup=0.4842x parity=0.000e+00
shm solo_stride1     165,232,369 ns  4.131 ns/update  lines(64B)=[0]
shm stride_1         342,173,261 ns  8.554 ns/update  lines(64B)=[0]
shm stride_8         310,046,809 ns  7.751 ns/update  lines(64B)=[0, 1, 2, 3]
shm stride_16        352,672,364 ns  8.817 ns/update  lines(64B)=[0, 2, 4, 6]
shm ratios: {'false_sharing_slowdown_stride1_vs_stride8': 1.1036, 'stride1_vs_stride16': 0.9702, 'stride8_vs_stride16': 0.8791}
LLVM: load i64=5 store i64=6 py_calls=[] nogil=True
kappa_max=0.240000 dt_max=3.958167e-06 dt=3.166533e-06
  step    0: E=1.063782502578e-01  max|u|=1.407969
  step  100: E=1.063633077817e-01  max|u|=1.380620
  step  200: E=1.063523752420e-01  max|u|=1.35428

### Analytical & Forensic Engineering Responses

> All numbers come from my run: GitHub `mikazhan`, Tesla T4 (CC 7.5) `GPU-175f6f16-84a4-9268-9b73-ae90379d451f`, host CPU Intel Xeon @ 2.00 GHz (2 logical CPUs = **1 physical core with 2 hyper-threads**, one shared 32 KiB L1d, 64-byte lines), `INSTANCE_SEED = 17370736032954436743 (0xf11140634b53cc87)`, `verification_passed = True`.

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):**
  The local field is a C-contiguous `(258, 258)` `float64` array, so consecutive elements of the halo column `u[1:257, 1]` are `258 × 8 = 2 064 B` apart.
  **On the host CPU** the smallest unit the cache moves is a 64-byte line. Each 8-byte column element therefore costs a full 64-byte line fill (8× overfetch): reading one column moves 256 lines = 16 384 B for 2 048 B of useful data. Successive lines are 2 064 B apart, so they don't help each other, and only a stride prefetcher can hide the latency.
  **On the T4** a warp's load is split into 32-byte L2/DRAM sectors (four sectors per 128-byte L1 line). A row read (`u[1, 1:257]`) by a warp is 32 consecutive doubles = 256 B = 8 sectors (100 % efficiency, fully coalesced). A column read by a warp touches 32 different rows → 32 distinct sectors = 1 024 B moved for 256 B used (4× overfetch, 25 % efficiency), in 32 separate transactions instead of 8.
  **PCIe/DRAM overhead avoided by `pack_halos_kernel`:** a single `cudaMemcpy` moves one contiguous byte range. The smallest contiguous range containing a column is `u[1,1] … u[256,1]` = `(255·258 + 1)·8 = 526 328 B` for 2 048 B of payload → **524 280 extra bytes per column (≈257×)**. Per step and rank that is 2 columns D2H + 2 ghost columns H2D = **2 097 120 extra bytes**, i.e. **629.1 MB per rank over 300 steps (2.52 GB for 4 ranks)**. My pack kernel does the strided reads once inside device memory (4× sector overfetch only on the GPU's ~300 GB/s DRAM) and writes the four strips into one contiguous 8 192-byte vector, so each step crosses PCIe only as one dense 8 KB D2H copy plus one 8 KB H2D copy into pinned buffers.

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):**
  `tiled_stencil_kernel.inspect_asm()` in my run: **st.shared.f64 = 5, ld.shared.f64 = 5, bar.sync = 1** (PTX audit PASS). The 5 stores are 1 center load into `smem[ty+1, tx+1]` plus the 4 conditional apron loads (top row by `ty==0`, bottom by `ty==15`, left by `tx==0`, right by `tx==15`). The 5 loads are the 5-point stencil (`c, i±1, j±1`). The single `bar.sync 0` is the one `cuda.syncthreads()` between tile loading and arithmetic.
  A 16 × 16 block is 256 threads = **8 warps**; warp *w* holds tile rows `ty = 2w, 2w+1`. A thread in row `ty` reads `smem[ty, ·]` (north neighbour) and `smem[ty+2, ·]` (south neighbour). For `ty = 2w` the north row was written by warp *w−1*; for `ty = 2w+1` the south row was written by warp *w+1*. The apron rows `smem[0, ·]` and `smem[17, ·]` are written only by warps 0 and 7. Warps are scheduled independently, so without the barrier a warp can execute its `ld.shared` before the producing warp's `st.shared` has happened. This is a cross-warp **Read-After-Write hazard**: the thread reads stale shared memory left by a previous block on that SM. The results become non-deterministic, errors appear along every 2-row warp boundary, and the serialized/overlapped parity check (exactly `0.000e+00` in my run) would fail. `bar.sync` guarantees all 256 threads' shared stores are complete and visible before any stencil load.

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):**
  Measured over 300 steps (max over ranks): **gpu_serialized_ms = 733.965** (CUDA events 733.849) vs **gpu_overlapped_ms = 1515.857** (CUDA events 1515.817) → speedup **0.4842×**. That is 2.45 ms vs 5.05 ms per step: the overlapped pipeline is 2.07× *slower*, while both produce identical fields (`max_diff = 0.000e+00`).
  The four MPI ranks are four OS processes, each with **its own CUDA context** on the single T4. Without MPS, the GPU time-slices between contexts: kernels from different ranks never run at the same time, and the copy engines and the one PCIe link are shared. All D2H/H2D DMA requests from the four contexts are therefore queued and serialized behind each other and behind other contexts' kernels. Within one context my two streams can overlap the interior kernel with the boundary DMA, but the overlap gain is tiny: halos are 4 × 2 KB through host shared memory (µs), and the interior kernel on 256 × 256 cells takes tens of µs. On the other side, the overlapped step costs 5 stencil launches instead of 1, two stream synchronisations, and extra context switches across the 4 processes. Those Python/launch overheads dominate, so the measured "overlap" is a net loss.
  With **four dedicated GPUs and GPUDirect RDMA**, each rank owns its GPU, so there is no time-slicing and each GPU has its own copy engines. The NIC also reads and writes GPU memory directly, so halos skip host staging, pinned copies and the shared PCIe queue. Communication then has real µs-level network latency that genuinely runs in parallel with the interior kernel on a different device. Posting `Isend/Irecv` first and computing the deep interior meanwhile hides that latency and gives a real speedup.

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):**
  `MPI.Win.Allocate_shared` window, 64 `uint64` slots, base address mod 64 = 0 and mod 128 = 0, 40 000 000 RMW updates per rank, median of 3:

  | Layout | Slots | 64-B lines touched | Latency (ns) | ns / update / rank |
  |---|---|---|---|---|
  | solo, stride 1 (rank 0 only) | [0] | [0] | 165,232,369 | 4.131 |
  | stride 1 (8 B) | [0, 1, 2, 3] | [0] | 342,173,261 | 8.554 |
  | stride 8 (64 B) | [0, 8, 16, 24] | [0, 1, 2, 3] | 310,046,809 | 7.751 |
  | stride 16 (128 B) | [0, 16, 32, 48] | [0, 2, 4, 6] | 352,672,364 | 8.817 |

  **False-sharing slowdown stride 1 / stride 8 = 1.104×**; stride 1 / stride 16 = 0.970×; stride 8 / stride 16 = 0.879×.
  *Interpretation of my numbers:* this Colab VM exposes only **2 logical CPUs, which are the two hyper-threads of one physical core** sharing one L1d. The 4 ranks are oversubscribed onto 2 hardware threads, so at most 2 ranks run simultaneously. The ideal time is ≈ 2 × solo = 330.5 ms, and all three layouts land within −6 % … +7 % of that. Because both hardware threads share the *same* L1, the line never moves between private caches, so there is **no inter-core RFO ping-pong here**. The 10 % stride-1 penalty comes from SMT-level effects (memory-ordering machine clears when the sibling thread stores to the same line, store-buffer contention). The stride-8 vs stride-16 ordering is within the scheduling noise of 4 processes time-sharing 2 threads. On a host with ≥ 4 physical cores the same code would show the classic multi-× penalty for stride 1.
  **MESI trace (Rank 0 → `shm_view[0]`, Rank 1 → `shm_view[1]`, both in line 0, ranks on different physical cores):**
  1. Initially the line is Invalid in both cores, or Shared after rank 0 zero-filled it and the Fence/Barrier.
  2. Core 0 stores `shm_view[0]`: it issues a **Read-For-Ownership**. Other copies go **S → I**; Core 0 gets the line **E** and, once the store commits, **M**.
  3. Core 1 stores `shm_view[1]`: its copy is **I**, so it issues its own RFO. Core 0 snoops and supplies the dirty line, then goes **M → I** (writing back or forwarding the data). Core 1 goes **I → E → M**.
  4. Core 0's next store misses again (**I**): RFO → Core 1 **M → I**, Core 0 **I → M**, and so on.

  Every hand-off is a cross-core coherence round trip, and the store at the head of the store buffer cannot commit until ownership arrives. The line ping-pongs although the ranks never touch the same 8-byte word. With 4 ranks at stride 1 all four cores contend for line 0; stride 8 gives each rank its own line, so steady state is M in one core with no RFOs.

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):**
  Measured: **stride 8 (64 B) = 310,046,809 ns vs stride 16 (128 B) = 352,672,364 ns** (ratio 0.879×). In my run the 128-byte layout is not faster; the 13.7 % difference is within the noise of 4 processes time-sliced onto 2 SMT threads of one core. On a single core the adjacent-line prefetcher pairs lines that live in the same private L2 anyway, so pairing cannot cause invalidations.
  On Intel cores the **L2 adjacent-cache-line (spatial) prefetcher** treats memory as 128-byte-aligned pairs of 64-byte lines: a miss on line 2k also fetches its buddy 2k+1, and vice versa. With 64-byte padding, rank 0's slot (line 0) and rank 1's slot (line 1) are buddies in the same 128-byte pair, as are ranks 2/3 (lines 2/3). When rank 0 misses, the prefetcher requests line 1, which is Modified in rank 1's core. That forces a snoop and steals or downgrades rank 1's copy, a **secondary invalidation** without any real sharing. This happens when (a) the padded slots sit in the same 128-B pair, (b) the prefetcher is enabled, and (c) the cores are physically distinct and write continuously. Stride 16 places each rank in its own pair (lines 0, 2, 4, 6 in my manifest), so prefetches only pull lines nobody else owns.
  **Padding trade-off:** padding wastes cache capacity. At stride 16 each 8-byte counter occupies 128 B, i.e. 6 % of the bytes are useful. For 4 counters this is irrelevant, but for per-thread arrays of counters, histograms or per-cell flags the footprint grows 16×. That pushes the working set out of the 32 KiB L1d (512 lines) and the 1 MiB L2, causing **capacity and conflict misses**, more TLB pressure, and more DRAM traffic than the false sharing it was meant to remove. The right padding is the coherency granule (64 B, or 128 B where adjacent-line prefetch or a 128-B line applies), and only for data that different cores actually write.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):**
  Δx = Δy = 1/513 = 1.949318×10⁻³, Δx² = 3.799840×10⁻⁶. Manifest `kappa_max = 0.240000` (κ ∈ [0.08000, 0.24000]), so
  **Δt_max = Δx² / (4·κ_max) = 3.958167×10⁻⁶**; the solver used **Δt = 3.166533×10⁻⁶ = 0.800·Δt_max**, so the Courant condition holds. For the 2-D 5-point scheme the amplification factor of the highest-frequency mode is G = 1 − 8κΔt/Δx² ≥ 1 − 1.6 = −0.6, so |G| ≤ 1 and the scheme is stable. The cubic term −λu³ only adds damping because λΔt·u² ≪ 1.
  Global energy E = Σ u·Δx·Δy via `cart_comm.allreduce(op=MPI.SUM)`:

  | step | E (allreduce) | ΔE | max\|u\| |
  |---|---|---|---|
  | 0 | 1.063782502578e-01 | — | 1.407969 |
  | 100 | 1.063633077817e-01 | −1.494e-05 | 1.380620 |
  | 200 | 1.063523752420e-01 | −1.093e-05 | 1.354282 |
  | 300 | 1.063418714281e-01 | −1.050e-05 | 1.328906 |

  E decreases monotonically (`energy_monotonic = True`, −3.42×10⁻⁴ relative over 300 steps), and ‖u‖₂ falls 0.2580389 → 0.2496260. The Dirichlet walls (ghost u = 0 on `MPI.PROC_NULL` sides) remove heat by boundary flux, and the cubic damping removes λΣu³ΔxΔyΔt per step; together they outweigh the small source injection. Serialized and overlapped modes print identical energy traces, and their final fields agree to `max_diff = 0.000e+00 < 10⁻¹²` on all 4 ranks.

